# IOAI — 2025 Contest Cluster Pictures (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/data_1.npz'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-cluster-pictures', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/data_1.npz' if os.path.exists('data/data_1.npz') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — Cluster Pictures

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (비지도 군집화)**

각 '사진'은 **rank-4 분해**로 두 행렬로 쪼개져 있습니다: `data_1[i]`(128×4) 와 `data_2[i]`(4×128).
원본 이미지 = `data_1[i] @ data_2[i]` (128×128). **3840장을 32개 군집으로** 비지도 군집화하는 과제입니다.

**채점**: 라벨 비공개라 **Submissions** 탭에서 `submission.csv`(`id,target`=군집번호)를 **캐글 리더보드에 자동 제출**해
점수를 받습니다(군집 일치도 지표, 높을수록 좋음).

⚠️ **아래 베이스라인 = 원시 인자(data_1·data_2)를 그대로 이어붙여 KMeans**(캐글 ≈ **0.028**, 거의 무작위 수준).
**함정**: 인자분해는 비유일(`data_1·M⁻¹` 와 `M·data_2` 도 같은 이미지) → 원시 인자는 군집 정보가 흐려집니다.
*재구성 이미지(곱)*로 군집화하면 오릅니다(모범답안 참고).

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
a = np.load('data/data_1.npz')['arr_0']   # (3840,128,4)
b = np.load('data/data_2.npz')['arr_0']   # (3840,4,128)
N, K = len(a), 32
print('items', N, '| clusters', K)


In [ ]:
# BASELINE: 원시 인자를 이어붙여 KMeans (비유일 표현 → 약함)
Xraw = np.concatenate([a.reshape(N, -1), b.reshape(N, -1)], axis=1)   # (3840, 1024)
labels = KMeans(K, n_init=10, random_state=0).fit_predict(Xraw)
pd.DataFrame({'id': range(N), 'target': labels}).to_csv('submission.csv', index=False)
print('wrote submission.csv | cluster sizes(head):', np.bincount(labels)[:6],
      '\n→ Submissions 탭에서 캐글 채점 (베이스라인 ≈ 0.028). 재구성 이미지로 개선하세요')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)